# Multimodal Multi-Task Training Pipeline

Kaggle/P100-ready PyTorch notebook matching `instruction.md`: each sample uses one RGB image and one synchronized FLAC audio recording. The model is trained from scratch, with image-only, audio-only, and image+audio ablations.

In [ ]:
from __future__ import annotations

import math
import os
import random
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn.functional as F
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score, mean_squared_error, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

ModalityMode = Literal['image_audio', 'image_only', 'audio_only']
FusionType = Literal['concat', 'gated', 'cross_attention']
AudioRep = Literal['log_mel', 'power_spectrogram', 'stft', 'raw_waveform']
DistanceNorm = Literal['divide_by_2', 'standardize']
DistanceLoss = Literal['mse', 'smooth_l1']

@dataclass
class ColumnConfig:
    image_name: str = 'image name'
    audio_name: str = 'audio name'
    object_type: str = 'Object_Type'
    distance: str = 'distance'
    location_zone: str = 'Location_Zone'
    illumination: str = 'Illumination'

@dataclass
class Config:
    train_csv: Path = Path('/kaggle/input/your-dataset/train.csv')
    test_csv: Path = Path('/kaggle/input/your-dataset/test.csv')
    image_dir: Path = Path('/kaggle/input/your-dataset/images')
    audio_dir: Path = Path('/kaggle/input/your-dataset/audios')
    mel_dir: Path = Path('/kaggle/working/mel')
    output_dir: Path = Path('/kaggle/working/multimodal_run')
    columns: ColumnConfig = field(default_factory=ColumnConfig)
    modality_mode: ModalityMode = 'image_audio'
    fusion: FusionType = 'concat'
    image_size: int = 224
    batch_size: int = 24
    epochs: int = 80
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 5
    patience: int = 12
    gradient_clip_norm: float = 1.0
    dropout: float = 0.3
    val_size: float = 0.2
    seed: int = 42
    deterministic: bool = True
    num_workers: int = 0  # Use 0 in Kaggle notebooks to avoid multiprocessing cleanup warnings.
    audio_representation: AudioRep = 'log_mel'
    overwrite_mel_cache: bool = False
    mel_cache_workers: int = 2
    audio_seconds: float = 2.0
    audio_target_sample_rate: int | None = None  # None avoids downsampling ultrasonic recordings.
    n_fft: int = 1024
    win_length: int | None = None
    hop_length: int = 256
    n_mels: int = 128
    audio_freq_bins: int = 128
    audio_time_bins: int = 256
    distance_normalization: DistanceNorm = 'divide_by_2'
    distance_divisor: float = 2.0
    distance_loss: DistanceLoss = 'mse'
    distance_clamp_min: float | None = None
    distance_clamp_max: float | None = None
    object_loss_weight: float = 0.40
    distance_loss_weight: float = 0.30
    zone_loss_weight: float = 0.20
    illumination_loss_weight: float = 0.10
    label_smoothing: float = 0.0
    use_class_weights: bool = False
    horizontal_flip: bool = True
    enable_train_augmentations: bool = True
    ema_decay: float = 0.0
    sanity_check: bool = False
    sanity_rows: int = 48

CFG = Config()

# Optional Kaggle CSV discovery. Uncomment to find the labeled training CSV.
# for path in sorted(Path('/kaggle/input').rglob('*.csv')):
#     df_preview = pd.read_csv(path, nrows=5)
#     print('\n', path)
#     print(df_preview.columns.tolist())
#     print(df_preview.head())

# Quick pipeline test without your dataset:
# CFG.sanity_check = True; CFG.epochs = 1; CFG.batch_size = 4; CFG.image_size = 64; CFG.audio_seconds = 0.5; CFG.num_workers = 0

# Kaggle note: if you see 'no kernel image is available for execution on the device'
# on Tesla P100, the installed PyTorch build does not support P100 sm_60.
# Switch Kaggle Accelerator to T4/L4/A100, or install a compatible PyTorch build before importing torch.
print(CFG)


## Dataset Inspection and Utilities

In [ ]:
def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    if not torch.cuda.is_available():
        print('CUDA is not available; using CPU.')
        return torch.device('cpu')
    name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print(f'CUDA device: {name}, compute capability: sm_{capability[0]}{capability[1]}')
    try:
        # Some Kaggle PyTorch builds no longer include kernels for Tesla P100 sm_60.
        x = torch.ones(1, device='cuda')
        _ = (x + 1).item()
        return torch.device('cuda')
    except Exception as exc:
        print('CUDA is visible but unusable with this PyTorch build; falling back to CPU.')
        print('Reason:', repr(exc))
        print('On Kaggle, switch Accelerator to T4/L4/A100, or install a PyTorch build that supports P100 sm_60 before importing torch and restart the session.')
        return torch.device('cpu')

def make_scheduler(optimizer: torch.optim.Optimizer, total_steps: int, warmup_steps: int) -> LambdaLR:
    total_steps = max(1, total_steps); warmup_steps = max(0, min(warmup_steps, total_steps - 1))
    def lr_lambda(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return LambdaLR(optimizer, lr_lambda)

def image_path(row: pd.Series, cfg: Config) -> Path:
    value = Path(str(row[cfg.columns.image_name]))
    return value if value.is_absolute() else cfg.image_dir / value

def audio_path(row: pd.Series, cfg: Config) -> Path:
    value = Path(str(row[cfg.columns.audio_name]))
    return value if value.is_absolute() else cfg.audio_dir / value

def read_and_validate_csv(path: Path, cfg: Config, require_targets: bool) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'CSV not found: {path}')
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    required = [cfg.columns.image_name, cfg.columns.audio_name]
    if require_targets:
        required += [cfg.columns.object_type, cfg.columns.distance, cfg.columns.location_zone, cfg.columns.illumination]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f'Missing columns {missing}. Available columns: {list(df.columns)}')
    for col in [cfg.columns.image_name, cfg.columns.audio_name, cfg.columns.object_type, cfg.columns.location_zone, cfg.columns.illumination]:
        if col in df.columns:
            df[col] = df[col].astype('string').str.strip().replace('', pd.NA)
    if require_targets:
        df[cfg.columns.distance] = pd.to_numeric(df[cfg.columns.distance], errors='coerce')
        target_cols = [cfg.columns.object_type, cfg.columns.distance, cfg.columns.location_zone, cfg.columns.illumination]
        null_counts = df[target_cols].isna().sum()
        print('Target null counts:'); print(null_counts)
        if null_counts.any():
            display(df[df[target_cols].isna().any(axis=1)].head(10))
            raise ValueError('Training targets contain NaN. CFG.train_csv must point to the labeled training CSV.')
    return df

def audio_metadata(path: Path) -> tuple[int, float, int]:
    if hasattr(torchaudio, 'info'):
        info = torchaudio.info(str(path))
        return int(info.sample_rate), float(info.num_frames / max(1, info.sample_rate)), int(info.num_channels)
    waveform, sr = torchaudio.load(str(path))
    return int(sr), float(waveform.size(1) / max(1, sr)), int(waveform.size(0))

def inspect_dataset(df: pd.DataFrame, cfg: Config, max_files: int = 40) -> None:
    print('Samples:', len(df))
    for col in [cfg.columns.object_type, cfg.columns.location_zone, cfg.columns.illumination]:
        if col in df.columns:
            print(f'\n{col} distribution:'); print(df[col].value_counts(dropna=False))
    if cfg.columns.distance in df.columns:
        print('\nDistance summary:'); print(df[cfg.columns.distance].describe())
    image_missing = sum(not image_path(row, cfg).exists() for _, row in df.head(max_files).iterrows())
    audio_missing = sum(not audio_path(row, cfg).exists() for _, row in df.head(max_files).iterrows())
    print(f'Checked first {min(max_files, len(df))} files: missing images={image_missing}, missing audio={audio_missing}')
    image_sizes, sample_rates, durations, channels = [], [], [], []
    for _, row in df.head(max_files).iterrows():
        ip = image_path(row, cfg); ap = audio_path(row, cfg)
        if ip.exists():
            with Image.open(ip) as img:
                image_sizes.append(img.size)
        if ap.exists():
            sr, duration, channel_count = audio_metadata(ap)
            sample_rates.append(sr); durations.append(duration); channels.append(channel_count)
    if image_sizes: print('Example image sizes:', pd.Series(image_sizes).value_counts().head())
    if sample_rates:
        print('Sample rates:', pd.Series(sample_rates).value_counts().head().to_dict())
        print('Audio duration seconds:', pd.Series(durations).describe())
        print('Audio channels:', pd.Series(channels).value_counts().to_dict())
        print('Max represented frequency is Nyquist = sample_rate / 2. No downsampling is applied by default.')

def visualize_audio_examples(df: pd.DataFrame, cfg: Config, n: int = 3) -> None:
    sample = df.sample(min(n, len(df)), random_state=cfg.seed)
    for _, row in sample.iterrows():
        path = audio_path(row, cfg)
        waveform, sr = torchaudio.load(str(path))
        mono = waveform.mean(dim=0)
        spec = torch.stft(mono, n_fft=cfg.n_fft, hop_length=cfg.hop_length, return_complex=True).abs().pow(2)
        log_spec = torch.log1p(spec)
        fig, axes = plt.subplots(1, 3, figsize=(15, 3))
        axes[0].plot(mono[: min(mono.numel(), sr)].numpy()); axes[0].set_title('waveform')
        axes[1].imshow(spec.numpy(), aspect='auto', origin='lower'); axes[1].set_title('power spectrogram')
        axes[2].imshow(log_spec.numpy(), aspect='auto', origin='lower'); axes[2].set_title('log spectrogram')
        plt.suptitle(str(path.name)); plt.show()

@dataclass
class CategoryMappings:
    class_to_index: dict[str, dict[str, int]]
    index_to_class: dict[str, dict[int, str]]

@dataclass
class DistanceNormalizer:
    method: str; divisor: float = 2.0; mean: float = 0.0; std: float = 1.0
    @classmethod
    def fit(cls, values: pd.Series, cfg: Config) -> 'DistanceNormalizer':
        arr = pd.to_numeric(values, errors='raise').astype(float).to_numpy()
        if cfg.distance_normalization == 'divide_by_2':
            return cls('divide_by_2', divisor=cfg.distance_divisor)
        mean = float(arr.mean()); std = float(arr.std())
        return cls('standardize', mean=mean, std=std if std > 1e-8 else 1.0)
    def normalize(self, arr: np.ndarray) -> np.ndarray:
        arr = arr.astype(np.float32)
        return arr / np.float32(self.divisor) if self.method == 'divide_by_2' else (arr - np.float32(self.mean)) / np.float32(self.std)
    def denormalize(self, arr: np.ndarray) -> np.ndarray:
        arr = arr.astype(np.float32)
        return arr * np.float32(self.divisor) if self.method == 'divide_by_2' else arr * np.float32(self.std) + np.float32(self.mean)

def fit_mappings(train_df: pd.DataFrame, cfg: Config) -> CategoryMappings:
    cols = {'object': cfg.columns.object_type, 'zone': cfg.columns.location_zone, 'illumination': cfg.columns.illumination}
    c2i, i2c = {}, {}
    for task, col in cols.items():
        labels = sorted(str(x) for x in train_df[col].dropna().unique())
        if not labels: raise ValueError(f'No labels found for {col}')
        c2i[task] = {label: i for i, label in enumerate(labels)}
        i2c[task] = {i: label for label, i in c2i[task].items()}
    return CategoryMappings(c2i, i2c)

def split_train_valid(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    target_cols = [cfg.columns.object_type, cfg.columns.distance, cfg.columns.location_zone, cfg.columns.illumination]
    if df[target_cols].isna().any().any():
        raise ValueError('Cannot split data with missing targets.')
    combined = df[cfg.columns.object_type].astype(str) + '__' + df[cfg.columns.location_zone].astype(str) + '__' + df[cfg.columns.illumination].astype(str)
    for name, labels in [('combined categorical targets', combined), ('Object_Type', df[cfg.columns.object_type].astype(str))]:
        try:
            tr, va = train_test_split(df, test_size=cfg.val_size, random_state=cfg.seed, shuffle=True, stratify=labels)
            print('Validation split stratification strategy:', name)
            return tr.reset_index(drop=True), va.reset_index(drop=True), name
        except ValueError as exc:
            print(f'Could not stratify by {name}: {exc}')
    tr, va = train_test_split(df, test_size=cfg.val_size, random_state=cfg.seed, shuffle=True)
    print('Validation split stratification strategy: random fallback')
    return tr.reset_index(drop=True), va.reset_index(drop=True), 'random fallback'


## Dataset and Preprocessing

In [ ]:
def make_image_transforms(cfg: Config, train: bool) -> transforms.Compose:
    mean = (0.485, 0.456, 0.406); std = (0.229, 0.224, 0.225)
    if train and cfg.enable_train_augmentations:
        ops: list[Any] = [transforms.RandomResizedCrop(cfg.image_size, scale=(0.75, 1.0), ratio=(0.85, 1.15))]
        if cfg.horizontal_flip: ops.append(transforms.RandomHorizontalFlip(p=0.5))
        ops += [transforms.ColorJitter(0.15, 0.15, 0.12, 0.03), transforms.RandomRotation(8), transforms.RandomAffine(0, translate=(0.03, 0.03)), transforms.ToTensor(), transforms.Normalize(mean, std)]
        return transforms.Compose(ops)
    return transforms.Compose([transforms.Resize(cfg.image_size + 32), transforms.CenterCrop(cfg.image_size), transforms.ToTensor(), transforms.Normalize(mean, std)])

def mel_path_from_audio_path(audio_file: Path, cfg: Config) -> Path:
    try:
        relative = audio_file.relative_to(cfg.audio_dir)
    except ValueError:
        relative = Path(audio_file.name)
    return (cfg.mel_dir / relative).with_suffix('.pt')

def mel_path(row: pd.Series, cfg: Config) -> Path:
    return mel_path_from_audio_path(audio_path(row, cfg), cfg)

def crop_or_pad_audio(waveform: torch.Tensor, sr: int, cfg: Config) -> torch.Tensor:
    target = max(1, int(round(cfg.audio_seconds * sr)))
    if waveform.size(1) >= target:
        return waveform[:, :target]
    return F.pad(waveform, (0, target - waveform.size(1)))

def compute_log_mel_tensor(audio_file: Path, cfg: Config) -> dict[str, Any]:
    waveform, sr = torchaudio.load(str(audio_file))
    original_sr = sr
    waveform = waveform.mean(dim=0, keepdim=True).float()
    if cfg.audio_target_sample_rate is not None and sr != cfg.audio_target_sample_rate:
        if cfg.audio_target_sample_rate < sr:
            print('Warning: downsampling can remove ultrasonic information:', audio_file.name)
        waveform = torchaudio.functional.resample(waveform, sr, cfg.audio_target_sample_rate)
        sr = cfg.audio_target_sample_rate
    waveform = crop_or_pad_audio(waveform, sr, cfg)
    mel = torchaudio.transforms.MelSpectrogram(sample_rate=sr, n_fft=cfg.n_fft, win_length=cfg.win_length, hop_length=cfg.hop_length, n_mels=cfg.n_mels, f_min=0.0, f_max=None, power=2.0)(waveform)
    mel = torchaudio.transforms.AmplitudeToDB(stype='power')(mel)
    mel = (mel - mel.mean()) / mel.std().clamp_min(1e-6)
    mel = F.interpolate(mel.unsqueeze(0), size=(cfg.audio_freq_bins, cfg.audio_time_bins), mode='bilinear', align_corners=False).squeeze(0).contiguous()
    return {'mel': mel.cpu(), 'source': str(audio_file), 'original_sample_rate': original_sr, 'sample_rate': sr, 'n_fft': cfg.n_fft, 'win_length': cfg.win_length, 'hop_length': cfg.hop_length, 'n_mels': cfg.n_mels}

def precompute_mel_cache(df: pd.DataFrame, cfg: Config) -> None:
    cfg.mel_dir.mkdir(parents=True, exist_ok=True)
    audio_files = sorted({audio_path(row, cfg) for _, row in df.iterrows()})
    def process_one(audio_file: Path) -> tuple[str, Path, str | None]:
        out_file = mel_path_from_audio_path(audio_file, cfg)
        if out_file.exists() and not cfg.overwrite_mel_cache:
            return 'skipped', audio_file, None
        if not audio_file.exists():
            return 'error', audio_file, 'missing file'
        try:
            payload = compute_log_mel_tensor(audio_file, cfg)
            out_file.parent.mkdir(parents=True, exist_ok=True)
            torch.save(payload, out_file)
            return 'created', audio_file, None
        except Exception as exc:
            return 'error', audio_file, repr(exc)
    errors: list[tuple[Path, str]] = []
    created = 0; skipped = 0
    workers = max(1, int(cfg.mel_cache_workers))
    if workers == 1:
        results = [process_one(path) for path in tqdm(audio_files, desc='precompute log-mel cache')]
    else:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = [pool.submit(process_one, path) for path in audio_files]
            results = [future.result() for future in tqdm(as_completed(futures), total=len(futures), desc='precompute log-mel cache')]
    for status, audio_file, err in results:
        if status == 'created': created += 1
        elif status == 'skipped': skipped += 1
        elif err is not None: errors.append((audio_file, err))
    print({'mel_created': created, 'mel_skipped': skipped, 'mel_errors': len(errors), 'mel_dir': str(cfg.mel_dir)})
    if errors:
        print('First mel preprocessing errors:')
        for path, err in errors[:20]:
            print(path, err)
        raise RuntimeError('Mel preprocessing failed. Fix missing/corrupt audio files before training.')

class MultimodalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cfg: Config, image_tf: transforms.Compose, mappings: CategoryMappings | None, normalizer: DistanceNormalizer | None, require_targets: bool) -> None:
        self.df = df.reset_index(drop=True).copy(); self.cfg = cfg; self.image_tf = image_tf; self.mappings = mappings; self.normalizer = normalizer; self.require_targets = require_targets
    def __len__(self) -> int: return len(self.df)
    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.df.iloc[idx]
        ip = image_path(row, self.cfg); ap = audio_path(row, self.cfg); mp = mel_path(row, self.cfg)
        if not ip.exists(): raise FileNotFoundError(f'Missing image file: {ip}')
        if not mp.exists(): raise FileNotFoundError(f'Missing cached mel file: {mp}. Run precompute_mel_cache before creating DataLoaders.')
        image = self.image_tf(Image.open(ip).convert('RGB'))
        try:
            payload = torch.load(mp, map_location='cpu', weights_only=True)
        except TypeError:
            payload = torch.load(mp, map_location='cpu')
        audio = payload['mel'] if isinstance(payload, dict) else payload
        sample: dict[str, Any] = {'image': image, 'audio': audio, 'image_name': str(row[self.cfg.columns.image_name]), 'audio_name': str(row[self.cfg.columns.audio_name])}
        if not self.require_targets: return sample
        assert self.mappings is not None and self.normalizer is not None
        raw_distance = np.array([float(row[self.cfg.columns.distance])], dtype=np.float32)
        sample.update({
            'object': torch.tensor(self.mappings.class_to_index['object'][str(row[self.cfg.columns.object_type])], dtype=torch.long),
            'distance': torch.tensor(self.normalizer.normalize(raw_distance)[0], dtype=torch.float32),
            'distance_raw': torch.tensor(raw_distance[0], dtype=torch.float32),
            'zone': torch.tensor(self.mappings.class_to_index['zone'][str(row[self.cfg.columns.location_zone])], dtype=torch.long),
            'illumination': torch.tensor(self.mappings.class_to_index['illumination'][str(row[self.cfg.columns.illumination])], dtype=torch.long),
        })
        return sample

def create_synthetic_dataset(cfg: Config) -> None:
    base = cfg.output_dir / 'sanity_data'; cfg.image_dir = base / 'images'; cfg.audio_dir = base / 'audio'; cfg.mel_dir = base / 'mel'; cfg.image_dir.mkdir(parents=True, exist_ok=True); cfg.audio_dir.mkdir(parents=True, exist_ok=True); cfg.mel_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(cfg.seed); objects = ['box', 'cone', 'cylinder']; zones = ['near', 'middle', 'far']; lights = ['low', 'normal']; rows = []
    sr = 48000
    for i in range(cfg.sanity_rows):
        image = np.zeros((96, 96, 3), dtype=np.uint8); image[:, :] = rng.integers(20, 220, size=3, dtype=np.uint8); image[10+i%40:34+i%40, 12+i%35:36+i%35] = rng.integers(120, 255, size=3, dtype=np.uint8)
        image_name = f'image_{i:04d}.png'; Image.fromarray(image).save(cfg.image_dir / image_name)
        t = torch.linspace(0, cfg.audio_seconds, int(sr * cfg.audio_seconds)); wave = 0.2 * torch.sin(2 * math.pi * (12000 + 300 * (i % 5)) * t).unsqueeze(0)
        audio_name = f'audio_{i:04d}.flac'; torchaudio.save(str(cfg.audio_dir / audio_name), wave, sr)
        rows.append({cfg.columns.image_name: image_name, cfg.columns.audio_name: audio_name, cfg.columns.object_type: objects[i % 3], cfg.columns.distance: 0.25 + (i % 8) * 0.25, cfg.columns.location_zone: zones[(i // 3) % 3], cfg.columns.illumination: lights[i % 2]})
    df = pd.DataFrame(rows); cfg.train_csv = base / 'train.csv'; cfg.test_csv = base / 'test.csv'; df.to_csv(cfg.train_csv, index=False); df[[cfg.columns.image_name, cfg.columns.audio_name]].iloc[:max(1, cfg.sanity_rows // 4)].to_csv(cfg.test_csv, index=False)


## Model: Image CNN + Audio CNN + Fusion + Four Heads

In [ ]:
class ResidualDownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.SiLU(inplace=True), nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch))
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.act = nn.SiLU(inplace=True); self.down = nn.MaxPool2d(2)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(self.act(self.conv(x) + self.skip(x)))

class CNNEncoder(nn.Module):
    def __init__(self, in_ch: int, embedding_dim: int = 256) -> None:
        super().__init__(); channels = [32, 64, 128, 256]; blocks = []; prev = in_ch
        for ch in channels: blocks.append(ResidualDownBlock(prev, ch)); prev = ch
        self.blocks = nn.Sequential(*blocks); self.pool = nn.AdaptiveAvgPool2d(1); self.proj = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], embedding_dim), nn.LayerNorm(embedding_dim), nn.SiLU(inplace=True))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(self.pool(self.blocks(x)))

class Fusion(nn.Module):
    def __init__(self, kind: FusionType, dim: int = 256) -> None:
        super().__init__(); self.kind = kind; self.dim = dim
        if kind == 'concat': self.out_dim = dim * 2
        elif kind == 'gated': self.gate = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid()); self.out_dim = dim
        elif kind == 'cross_attention': self.attn = nn.MultiheadAttention(dim, num_heads=4, batch_first=True); self.out_dim = dim * 2; self.last_weights: torch.Tensor | None = None
        else: raise ValueError(kind)
    def forward(self, image_emb: torch.Tensor, audio_emb: torch.Tensor) -> torch.Tensor:
        if self.kind == 'concat': return torch.cat([image_emb, audio_emb], dim=1)
        if self.kind == 'gated':
            gate = self.gate(torch.cat([image_emb, audio_emb], dim=1)); return gate * image_emb + (1.0 - gate) * audio_emb
        tokens = torch.stack([image_emb, audio_emb], dim=1); attended, weights = self.attn(tokens, tokens, tokens, need_weights=True); self.last_weights = weights.detach().cpu(); return attended.flatten(1)

class MultimodalNet(nn.Module):
    def __init__(self, cfg: Config, num_object: int, num_zone: int, num_illumination: int) -> None:
        super().__init__(); self.cfg = cfg; emb = 256
        self.image_encoder = CNNEncoder(3, emb)
        self.audio_encoder = CNNEncoder(1, emb)
        self.fusion = Fusion(cfg.fusion, emb)
        if cfg.modality_mode == 'image_audio': fused_dim = self.fusion.out_dim
        elif cfg.modality_mode in ('image_only', 'audio_only'): fused_dim = emb
        else: raise ValueError(cfg.modality_mode)
        self.shared = nn.Sequential(nn.Linear(fused_dim, 512), nn.LayerNorm(512), nn.SiLU(inplace=True), nn.Dropout(cfg.dropout))
        self.object_head = nn.Linear(512, num_object); self.distance_head = nn.Linear(512, 1); self.zone_head = nn.Linear(512, num_zone); self.illumination_head = nn.Linear(512, num_illumination)
        self._init_weights()
    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear): nn.init.trunc_normal_(m.weight, std=0.02); nn.init.zeros_(m.bias) if m.bias is not None else None
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def extract_features(self, image: torch.Tensor, audio: torch.Tensor) -> torch.Tensor:
        image_emb = self.image_encoder(image); audio_emb = self.audio_encoder(audio)
        if self.cfg.modality_mode == 'image_only': return image_emb
        if self.cfg.modality_mode == 'audio_only': return audio_emb
        return self.fusion(image_emb, audio_emb)
    def forward(self, image: torch.Tensor, audio: torch.Tensor) -> dict[str, torch.Tensor]:
        h = self.shared(self.extract_features(image, audio)); distance = self.distance_head(h).squeeze(1)
        assert distance.shape == (image.shape[0],)
        return {'object': self.object_head(h), 'distance': distance, 'zone': self.zone_head(h), 'illumination': self.illumination_head(h)}

class MultiTaskLoss(nn.Module):
    def __init__(self, cfg: Config, class_weights: dict[str, torch.Tensor | None]) -> None:
        super().__init__(); self.cfg = cfg
        self.object_ce = nn.CrossEntropyLoss(weight=class_weights.get('object'), label_smoothing=cfg.label_smoothing)
        self.zone_ce = nn.CrossEntropyLoss(weight=class_weights.get('zone'), label_smoothing=cfg.label_smoothing)
        self.illum_ce = nn.CrossEntropyLoss(weight=class_weights.get('illumination'), label_smoothing=cfg.label_smoothing)
        self.distance_loss = nn.MSELoss() if cfg.distance_loss == 'mse' else nn.SmoothL1Loss()
    def forward(self, outputs: dict[str, torch.Tensor], targets: dict[str, torch.Tensor]) -> tuple[torch.Tensor, dict[str, float], dict[str, float]]:
        raw_t = {'object': self.object_ce(outputs['object'], targets['object']), 'distance': self.distance_loss(outputs['distance'], targets['distance']), 'zone': self.zone_ce(outputs['zone'], targets['zone']), 'illumination': self.illum_ce(outputs['illumination'], targets['illumination'])}
        weights = {'object': self.cfg.object_loss_weight, 'distance': self.cfg.distance_loss_weight, 'zone': self.cfg.zone_loss_weight, 'illumination': self.cfg.illumination_loss_weight}
        total = sum(weights[k] * raw_t[k] for k in raw_t)
        return total, {k: float(v.detach().cpu()) for k, v in raw_t.items()}, {k: float((weights[k] * raw_t[k]).detach().cpu()) for k in raw_t}

class ModelEMA:
    def __init__(self, model: nn.Module, decay: float) -> None:
        self.decay = decay; self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items() if torch.is_floating_point(v)}
    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        state = model.state_dict()
        for k, v in self.shadow.items(): v.mul_(self.decay).add_(state[k].detach(), alpha=1.0 - self.decay)
    def apply(self, model: nn.Module) -> dict[str, torch.Tensor]:
        backup = {}; state = model.state_dict()
        for k, v in self.shadow.items(): backup[k] = state[k].detach().clone(); state[k].copy_(v)
        return backup
    def restore(self, model: nn.Module, backup: dict[str, torch.Tensor]) -> None:
        state = model.state_dict()
        for k, v in backup.items(): state[k].copy_(v)


## Prepare Data

In [ ]:
set_seed(CFG.seed, CFG.deterministic)
if CFG.sanity_check:
    CFG.output_dir = Path('/kaggle/working/multimodal_sanity') if Path('/kaggle/working').exists() else Path(tempfile.mkdtemp(prefix='multimodal_sanity_'))
    create_synthetic_dataset(CFG)
if not CFG.train_csv.exists(): raise FileNotFoundError(f'Train CSV not found: {CFG.train_csv}')
if not CFG.image_dir.exists(): raise FileNotFoundError(f'Image directory not found: {CFG.image_dir}')
if not CFG.audio_dir.exists(): raise FileNotFoundError(f'Audio directory not found: {CFG.audio_dir}')

full_df = read_and_validate_csv(CFG.train_csv, CFG, require_targets=True)
display(full_df.head()); inspect_dataset(full_df, CFG)
# visualize_audio_examples(full_df, CFG, n=3)
cache_frames = [full_df[[CFG.columns.image_name, CFG.columns.audio_name]]]
if CFG.test_csv.exists():
    test_cache_df = read_and_validate_csv(CFG.test_csv, CFG, require_targets=False)
    cache_frames.append(test_cache_df[[CFG.columns.image_name, CFG.columns.audio_name]])
cache_df = pd.concat(cache_frames, ignore_index=True).drop_duplicates(subset=[CFG.columns.audio_name])
precompute_mel_cache(cache_df, CFG)
train_df, valid_df, split_strategy = split_train_valid(full_df, CFG)
mappings = fit_mappings(train_df, CFG); normalizer = DistanceNormalizer.fit(train_df[CFG.columns.distance], CFG)
num_object = len(mappings.class_to_index['object']); num_zone = len(mappings.class_to_index['zone']); num_illumination = len(mappings.class_to_index['illumination'])
print('Classes:', {'object': num_object, 'zone': num_zone, 'illumination': num_illumination})
print('Distance normalizer:', normalizer)

train_ds = MultimodalDataset(train_df, CFG, make_image_transforms(CFG, True), mappings, normalizer, True)
valid_ds = MultimodalDataset(valid_df, CFG, make_image_transforms(CFG, False), mappings, normalizer, True)
device = get_device(); use_amp = device.type == 'cuda'
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=use_amp, drop_last=False)
valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=use_amp, drop_last=False)
print('Device:', device, 'AMP:', use_amp)

model = MultimodalNet(CFG, num_object, num_zone, num_illumination).to(device)
batch = next(iter(train_loader))
with torch.no_grad():
    out = model(batch['image'].to(device), batch['audio'].to(device))
print('Output shapes:', {k: tuple(v.shape) for k, v in out.items()})


## Train and Validate

In [ ]:
def batch_to_device(batch: dict[str, Any], device: torch.device) -> tuple[torch.Tensor, torch.Tensor, dict[str, torch.Tensor]]:
    image = batch['image'].to(device, non_blocking=True); audio = batch['audio'].to(device, non_blocking=True)
    targets = {'object': batch['object'].to(device, non_blocking=True).long(), 'distance': batch['distance'].to(device, non_blocking=True).float(), 'zone': batch['zone'].to(device, non_blocking=True).long(), 'illumination': batch['illumination'].to(device, non_blocking=True).long()}
    return image, audio, targets

def class_weights() -> dict[str, torch.Tensor | None]:
    if not CFG.use_class_weights: return {'object': None, 'zone': None, 'illumination': None}
    def weights(col: str, task: str, n: int) -> torch.Tensor:
        labels = torch.tensor([mappings.class_to_index[task][str(x)] for x in train_df[col]], dtype=torch.long)
        counts = torch.bincount(labels, minlength=n).float().clamp_min(1)
        return (counts.sum() / (n * counts)).to(device)
    return {'object': weights(CFG.columns.object_type, 'object', num_object), 'zone': weights(CFG.columns.location_zone, 'zone', num_zone), 'illumination': weights(CFG.columns.illumination, 'illumination', num_illumination)}

def compute_metrics(y_true: dict[str, np.ndarray], y_pred: dict[str, np.ndarray]) -> dict[str, float]:
    pred_distance = normalizer.denormalize(y_pred['distance'])
    rmse = float(np.sqrt(mean_squared_error(y_true['distance_raw'], pred_distance)))
    dscore = max(0.0, 1.0 - rmse / 2.0)
    obj = float(f1_score(y_true['object'], y_pred['object'], average='macro', zero_division=0)); zone = float(f1_score(y_true['zone'], y_pred['zone'], average='macro', zero_division=0)); illum = float(f1_score(y_true['illumination'], y_pred['illumination'], average='macro', zero_division=0))
    score = 100.0 * (0.40 * obj + 0.30 * dscore + 0.20 * zone + 0.10 * illum)
    return {'object_macro_f1': obj, 'zone_macro_f1': zone, 'illumination_macro_f1': illum, 'distance_rmse': rmse, 'distance_score': dscore, 'competition_score': score}

def grad_norm(model: nn.Module) -> float:
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += float(p.grad.detach().norm(2).cpu()) ** 2
    return math.sqrt(total)

def train_one_epoch() -> dict[str, float]:
    model.train(); totals = {'loss': 0.0, 'object_loss': 0.0, 'distance_loss': 0.0, 'zone_loss': 0.0, 'illumination_loss': 0.0, 'grad_norm': 0.0}; seen = 0
    for batch in tqdm(train_loader, desc='train', leave=False):
        image, audio, targets = batch_to_device(batch, device); bs = image.size(0); optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(image, audio); loss, raw, weighted = criterion(outputs, targets)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        gn = grad_norm(model)
        if CFG.gradient_clip_norm > 0: nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip_norm)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        if ema is not None: ema.update(model)
        seen += bs; totals['loss'] += float(loss.detach().cpu()) * bs; totals['grad_norm'] += gn * bs
        for k, v in raw.items(): totals[f'{k}_loss'] += v * bs
    return {k: v / max(1, seen) for k, v in totals.items()}

@torch.no_grad()
def validate() -> tuple[dict[str, float], dict[str, np.ndarray], dict[str, np.ndarray]]:
    model.eval(); totals = {'val_loss': 0.0, 'val_object_loss': 0.0, 'val_distance_loss': 0.0, 'val_zone_loss': 0.0, 'val_illumination_loss': 0.0}; seen = 0
    yt = {'object': [], 'zone': [], 'illumination': [], 'distance_raw': []}; yp = {'object': [], 'zone': [], 'illumination': [], 'distance': []}
    for batch in tqdm(valid_loader, desc='valid', leave=False):
        image, audio, targets = batch_to_device(batch, device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(image, audio); loss, raw, weighted = criterion(outputs, targets)
        bs = image.size(0); seen += bs; totals['val_loss'] += float(loss.detach().cpu()) * bs
        for k, v in raw.items(): totals[f'val_{k}_loss'] += v * bs
        yt['object'].append(targets['object'].cpu().numpy()); yt['zone'].append(targets['zone'].cpu().numpy()); yt['illumination'].append(targets['illumination'].cpu().numpy()); yt['distance_raw'].append(batch['distance_raw'].numpy().astype(np.float32))
        yp['object'].append(outputs['object'].argmax(1).cpu().numpy()); yp['zone'].append(outputs['zone'].argmax(1).cpu().numpy()); yp['illumination'].append(outputs['illumination'].argmax(1).cpu().numpy()); yp['distance'].append(outputs['distance'].cpu().numpy().astype(np.float32))
    avg = {k: v / max(1, seen) for k, v in totals.items()}; yt_np = {k: np.concatenate(v) for k, v in yt.items()}; yp_np = {k: np.concatenate(v) for k, v in yp.items()}
    return {**avg, **compute_metrics(yt_np, yp_np)}, yt_np, yp_np

criterion = MultiTaskLoss(CFG, class_weights()).to(device)
optimizer = AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
total_steps = max(1, len(train_loader) * CFG.epochs); warmup_steps = min(total_steps - 1, len(train_loader) * CFG.warmup_epochs)
scheduler = make_scheduler(optimizer, total_steps, warmup_steps)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
ema = ModelEMA(model, CFG.ema_decay) if CFG.ema_decay > 0 else None
history: list[dict[str, Any]] = []; best_score = -float('inf'); bad_epochs = 0; best_true = None; best_pred = None
CFG.output_dir.mkdir(parents=True, exist_ok=True)
for epoch in range(1, CFG.epochs + 1):
    train_stats = train_one_epoch()
    if ema is not None:
        backup = ema.apply(model); val_stats, y_true, y_pred = validate(); ema.restore(model, backup)
    else:
        val_stats, y_true, y_pred = validate()
    row = {'epoch': epoch, 'lr': float(optimizer.param_groups[0]['lr']), **train_stats, **val_stats}; history.append(row); pd.DataFrame(history).to_csv(CFG.output_dir / 'training_history.csv', index=False)
    print(f"epoch {epoch:03d} | train {row['loss']:.4f} | val {row['val_loss']:.4f} | obj {row['object_macro_f1']:.4f} | zone {row['zone_macro_f1']:.4f} | illum {row['illumination_macro_f1']:.4f} | rmse {row['distance_rmse']:.4f} | score {row['competition_score']:.4f}")
    payload = {'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(), 'scaler_state': scaler.state_dict(), 'epoch': epoch, 'best_score': max(best_score, row['competition_score']), 'config': asdict(CFG), 'class_to_index': mappings.class_to_index, 'index_to_class': {t: {str(k): v for k, v in d.items()} for t, d in mappings.index_to_class.items()}, 'distance_normalization': asdict(normalizer), 'model_config': {'num_object': num_object, 'num_zone': num_zone, 'num_illumination': num_illumination}, 'training_history': history}
    torch.save(payload, CFG.output_dir / 'last_model.pt')
    if row['competition_score'] > best_score:
        best_score = float(row['competition_score']); payload['best_score'] = best_score; torch.save(payload, CFG.output_dir / 'best_model.pt'); best_true = y_true; best_pred = y_pred; bad_epochs = 0; print('  saved best_model.pt')
    else:
        bad_epochs += 1
        if bad_epochs >= CFG.patience: print('Early stopping.'); break


## Analysis Plots

In [ ]:
def analyze_validation(y_true: dict[str, np.ndarray], y_pred: dict[str, np.ndarray]) -> None:
    history_df = pd.DataFrame(history)
    if not history_df.empty:
        history_df[['loss', 'val_loss', 'competition_score']].plot(subplots=True, figsize=(10, 8)); plt.show()
    for task, labels in [('object', mappings.index_to_class['object']), ('zone', mappings.index_to_class['zone']), ('illumination', mappings.index_to_class['illumination'])]:
        cm = confusion_matrix(y_true[task], y_pred[task], labels=list(labels.keys()))
        ConfusionMatrixDisplay(cm, display_labels=[labels[i] for i in labels]).plot(xticks_rotation=45); plt.title(task); plt.show()
        p, r, f, s = precision_recall_fscore_support(y_true[task], y_pred[task], labels=list(labels.keys()), zero_division=0)
        display(pd.DataFrame({'class': [labels[i] for i in labels], 'precision': p, 'recall': r, 'f1': f, 'support': s}))
    errors = normalizer.denormalize(y_pred['distance']) - y_true['distance_raw']
    plt.hist(errors, bins=30); plt.title('Distance prediction error'); plt.show()

@torch.no_grad()
def embedding_visualization(max_batches: int = 4) -> None:
    model.eval(); feats, labels = [], []
    for i, batch in enumerate(valid_loader):
        if i >= max_batches: break
        image, audio, targets = batch_to_device(batch, device)
        feats.append(model.extract_features(image, audio).cpu().numpy()); labels.append(targets['object'].cpu().numpy())
    x = np.concatenate(feats); y = np.concatenate(labels); xy = PCA(n_components=2).fit_transform(x)
    plt.scatter(xy[:, 0], xy[:, 1], c=y, cmap='tab10', s=12); plt.title('Validation embeddings PCA'); plt.show()

if best_true is not None and best_pred is not None:
    analyze_validation(best_true, best_pred)
    embedding_visualization()


## Submission

In [ ]:
@torch.no_grad()
def create_submission(checkpoint_path: Path, test_csv: Path, output_path: Path) -> pd.DataFrame:
    ckpt = torch.load(checkpoint_path, map_location=device)
    index_to_class = {t: {int(k): v for k, v in d.items()} for t, d in ckpt['index_to_class'].items()}
    dist_norm = DistanceNormalizer(**ckpt['distance_normalization'])
    model.load_state_dict(ckpt['model_state']); model.eval()
    test_df = read_and_validate_csv(test_csv, CFG, require_targets=False)
    precompute_mel_cache(test_df[[CFG.columns.image_name, CFG.columns.audio_name]], CFG)
    test_ds = MultimodalDataset(test_df, CFG, make_image_transforms(CFG, False), None, None, False)
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size * 2, shuffle=False, num_workers=CFG.num_workers, pin_memory=use_amp, drop_last=False)
    rows = []
    for batch in tqdm(test_loader, desc='predict', leave=False):
        image = batch['image'].to(device, non_blocking=True); audio = batch['audio'].to(device, non_blocking=True); out = model(image, audio)
        dist = dist_norm.denormalize(out['distance'].cpu().numpy())
        if CFG.distance_clamp_min is not None or CFG.distance_clamp_max is not None: dist = np.clip(dist, CFG.distance_clamp_min, CFG.distance_clamp_max)
        obj = out['object'].argmax(1).cpu().numpy(); zone = out['zone'].argmax(1).cpu().numpy(); illum = out['illumination'].argmax(1).cpu().numpy()
        for image_name, audio_name, o, d, z, il in zip(batch['image_name'], batch['audio_name'], obj, dist, zone, illum):
            rows.append({CFG.columns.image_name: image_name, CFG.columns.audio_name: audio_name, CFG.columns.object_type: index_to_class['object'][int(o)], CFG.columns.distance: float(d), CFG.columns.location_zone: index_to_class['zone'][int(z)], CFG.columns.illumination: index_to_class['illumination'][int(il)]})
    sub = pd.DataFrame(rows); output_path.parent.mkdir(parents=True, exist_ok=True); sub.to_csv(output_path, index=False); return sub

if CFG.test_csv.exists():
    submission_path = Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else CFG.output_dir / 'submission.csv'
    submission = create_submission(CFG.output_dir / 'best_model.pt', CFG.test_csv, submission_path)
    display(submission.head()); print('Saved', submission_path)
else:
    print('No test CSV found; skipped submission.')
